# The incremental load that loses late events

**Track:** pyspark | **Difficulty:** medium | **Tags:** incremental load, late-arriving data, watermark

Run the three setup cells below once, in order. Then read the problem, write your
answer in the answer cell so that it ends with a DataFrame called `result`, and run
`check(result)`. You can run the check as many times as you like.

In [ ]:
# Setup: installs PySpark on Colab and starts a local Spark session (about a minute).
import os
import re
import shutil
import subprocess
import sys

PYSPARK_VERSION = "4.1.1"

if "google.colab" in sys.modules:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", f"pyspark=={PYSPARK_VERSION}"],
        check=True,
    )
    # PySpark 4 needs Java 17 or newer.
    java_major = 0
    if shutil.which("java"):
        version_text = subprocess.run(["java", "-version"], capture_output=True, text=True).stderr
        match = re.search(r'version "(\d+)', version_text)
        java_major = int(match.group(1)) if match else 0
    if java_major < 17:
        subprocess.run(["apt-get", "install", "-y", "-qq", "openjdk-17-jdk-headless"], check=True)
        os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark = (
    SparkSession.builder.master("local[2]")
    .appName("003-incremental-load-late-events")
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.ui.enabled", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version, "is ready.")

In [ ]:
#@title Load the data (run this cell, no need to read it)
import base64
import pathlib

_DATA = {
    "events": """
UEFSMRUEFfAWFdoLTBXuAhUAEgAAuAsIwiMABQEAiQ0IAGINCADWDQgAtA0IALYNCABEDQgAdw0IAEgNCACVDQgAqg0IAFwNCAB5
DQgAeA0IAGkNCACyDQgAhQ0IAKINCACWDQgAYA0IANsNCADZDQgAOA0IAGsNCACMDQgAbQ0IAEMNCAB7DQgAow0IAL0NCABRDQgA
bw0IAJANCADVDQgAXQ0IACkNCADBDQgAcw0IAJMNCACHDQgAug0IALwNCADaDQgAcA0IAL8NCAA+DQgAWA0IADsNCACNDQgArQ0I
ADMNCAC7DQgAZg0IALgNCABKDQgAxg0IAIANCABLDQgAgw0IADENCAQeJSXfBAA/DRAAXg0IAFINCACGDQgEviMJKAA5DQgAzQ0I
AC0NCACbDQgA3A0IAC4NCAA1DQgAyg0IAJQNCABBDQgARQ0IAKYNCABGDQgAoA0IAGgNCAB2DQgALw0IAHENCADODQgATA0IAFsN
CADTDQgAKw0IALkNCACCDQgAtw0IAIsNCABWDQgA1w0IAFkNCABHDQgANg0IAK4NCABaDQgAww0IAMUNCADLDQgApw0IAIENCABf
DQgAHS1wAE8NEADSDQgAYQ0IAMwNCAC1DQgAbg0IANgNCACsDQgApA0IAKsNCACPDQgAnA0IACoNCAB+DQgAng0IAGoNCACfDQgA
ZA0IAB8NmACoDRAA0Q0IANQNCABVDQgAVA0IAFcNCACSDQgAsA0IAGcNCABODQgAlw0IAIgNCACEDQgAdA0IADcNCAB8DQgANA0I
ADANCAB1DQgEUyNJgACaDQgAxA0IAJ0NCACYDQgAQA0IAGMNCAByDQgAZQ0IADoNCADHDQgAMg0IALMNCAClDQgAyQ0IAHoNCACK
DQgAsQ0IAD0NCADPDQgAjg0IAMgNCAB/DQgAbA0IAKENCAA8DQgAUA0IANANCACRDQgATQ0IACwNCACvDQgAmQ0IAEINCAB9DQgA
wA0IPKkjAAAAAAAASSMAAAAAAAAVABWiAxWqAywVggMVEBUGFQYcGAgfJQAAAAAAABgIKSMAAAAAAAAWACgIHyUAAAAAAAAYCCkj
AAAAAAAAEREAAADRAfDQAwAAAIIDAQgzAAECAwQFBgcICQoLDA0ODxAREhMUFRYXGAUZBBobHB0eHyAhIiMkJSYnKCkqKxssLS4v
MDEyMzQ1Njc4OTo7PD0+P0BBQkNERUZHSElKS0wBTU5PUFFSU1RVVldYWVpbXF1eX2BhYmNkZWZnaGlqazNsbW5vcHFyc3R1dnd4
XHl6e3x9fjR/gIGCg4SFhoeIiYqLjI2Oj5CRkpOUlZaXmJmam5ydnp+goaIUo6SlpqeoqaqrrK2ur5SwX7Gys7S1tgAAAAAAAAAV
BBWABRXeAkwVUBUAEgAAwAIEHwAJAQAnCQcEAAENCAAIDQgAIg0IAAkNCAAjDQgAAw0IABENCAALDQgADA0IABwNCAAGDQgAFQ0I
ACUNCAACDQgAEA0IABINCAAZDQgADw0IABQNCAAWDQgABQ0IAAcNCAAkDQgAHg0IABoNCAAXDQgAKA0IABMNCAAEDQgEHQAJAQAK
CQcEAA4NCAAmDQgAIA0IABgNCAAbDQg8IQAAAAAAAAANAAAAAAAAABUAFb4CFcYCLBWCAxUQFQYVBhwYCCgAAAAAAAAAGAgBAAAA
AAAAABYAKAgoAAAAAAAAABgIAQAAAAAAAAAREQAAAJ8B8J4DAAAAggMBBjNAIAxEYRxIoijIwjTO8ECRNC1EsRBUZQUXlmEalGyB
MjBWUnFBh3WY5BXORwUPxRkEB4FUWCWHWFnOmJBBxGjhsFWlB0STYnYfMIJF9okZCEQmAZYeSS3TBpxXFXKM8iWFCG1RYQzCJ1Eb
aBQZpGFmMmzLIw7NYEpURk0nGJxLFHkY81EgRG6NZJUDBEYmxnAUAAAAAAAVBBVkFWZMFQgVABIAADJsCwAAAGFkZF90b19jYXJ0
CAAAAHB1cmNoYXNlCQEMRGFnZV92aWV3BgAAAHNlYXJjaBUAFXYVeiwVggMVEBUGFQYcNgAoBnNlYXJjaBgLYWRkX3RvX2NhcnQR
EQAAADvoAwAAAIIDAQIzUKCVmhuuMcOWLueMxjMIT7hB5hrkQP3KzKcS0Vec4YEIkWpxx1D7gx4HoS56+nN/AQAVBBXwFhWCFkwV
7gIVABIAALgL8DxApP1WoC8GAEDl1wquLwYAwCG2y4kvBgBAS11uwy8GAMCfGNGyLwYAgLvGaWgvBgDAzxrrYS8GAEBsHjNoAQhs
8sqjsC8GAIC0PYqbLwYAQPpyu44vBgCAMqB9SgEITPRbpZovBgBAsOmypi8GAMAXtP63AQj0ggItXft5LwYAQO1kYk4vBgAA4pQ7
hi8GAMB2ZcNbLwYAALaARHwvBgCAGozooS8GAIA42Bt4LwYAwNZZxosvBgCAIYANZC8GAEBOIfd4LwYAwPGuvpYvBgCAyb2MuC8G
AABWxOalLwYAQJPjXpEvBgBAZeGFWS8GAMD1hPigLwYAgKg9hnkvBgAA6+XjoS8GAEAcvgZ3LwYAgFM7Mk0vBgDARt/IVy8GAAAX
NSp3LwYAgODLhK8vBgDA8Xnnoi8GAADNVqBWLwYAgNWJPnovBgBAwYgHYS8GAECyuq+4LwYAANs2VmIvBgCAYe0ohy8GAEBUOeth
LwYAgB7+65svBgCAIKNFfS8GAMAihgN3LwYAANZth3cvBgDAmqAVUC8GAABr426eLwYAgDVKAcQvBgDALpFytC8GAMCva8WXLwYA
QJhrVF8vBgAAihLrby8GAMDwuyB5LwYAwOZZZ1MvBgDAdqPXTy8GAABc1ll6LwYAQMKYipovBgDAXR23eC8GAMDOg+aeLwYAgL1N
WZovBgBAx3cvvC8GAAC2eJ6JLwYAgOJM4VIvBgBApzNGqi8GAAC8NAuSLwYAgKQcqIEvBgCAvk+sjy8GAACsFVeTLwYAQB4560Iv
BgBARPzMsy8GAMBMmqePLwYAgGtsvosvBgAAxS+bZy8GAMAIZhB3LwYAgP31w4gvBgCAqGZkgS8GAIDBHIS/LwYAwP5LP1MvBgCA
tBVDoS8GAAB8HyRWLwYAgIQ7XnIvBgCALfUqiS8GAECYNoaoLwYAgOcRXUQvBgCA9fgzry8GAIDfYAhmLwYAgHdgt0cvBgBAjAt2
Yy8GAEA3WlB2LwYAABFPSFkvBgBAtS9KIVgQQKEIQbMBMCihaQK/LwYAwOp01kHQUAD04QqFLwYAQI6nkkQvBgCAE9dlpAEILM4T
Z34vBgDAhUQZrgEICJNrIAEITEBTO/SiLwYAAPz+O2YvBgDAYZgAAUgMQDyhfkHAVMDORDudLwYAAP1zg4AvBgBAMo0lVS8BEETx
0EYvBgCAwSDXuC8GAACWbw0BIGyAmGaxPy8GAMAjIWRwLwYAQM2TbWEvBgDANd/5YQhMAHvMq14vBgCA/aJway8GAMD/RyMBiAwA
Yi8nAUBMAML+lWQvBgCAtizxXC8GAAAchZUFoAj7HIoFQAxgnEefARhI+Xrsgi8GAED3afRILwYAwFC2mwFATEC8sIOeLwYAgJZ5
eJkvBgBAomeyYWAswAjvyFgvBgAA8XqXASgUgMSHOEMvISgESd0BQEyALEE4bS8GAMDEZCt0LwYAAFVggiUwCMvCB2HYTMDg0Uyr
LwYAQDBVNkcvBgAAGlmfAUgMAHa2CgHIUECtmjF6LwYAgDqjl3UvBgAA7bbXdgEIKE+1+GgvBgDAxkergZgsQCt1A1UvBgCAGyAx
gUgMgG9XLgHIDMDQ1oMlMCj7utGnLwYAQOG1jAFYEICFLW6/ARAMq2UHlQEICBv7VSF4DMDzQBABKEzA/3inqC8GAADh9PnCLwYA
wBxmYCVgCDZI+AFYDEDe0vAByBSAAwYSfy+BsARXviVADIfJ6lcFEEh1KE8vBgBAMcYzpS8GAMCMAKJTAQgI60WlAfBMgNZFzksv
BgCAKC8PTS8GAAD8AyYhKAxAU31SASgMgIBYnQUoEOZ1ZT0vIdAEPU+lmBC4Z1VvLwFYNCEkeC8GAIDtClBwLwYAFQAVogMVqgMs
FYIDFRAVBhUGHBgIQLUvSsQvBgAYCIDmdWU9LwYAFgAoCEC1L0rELwYAGAiA5nVlPS8GABERAAAA0QHw0AMAAACCAwEIMwABAgME
BQYHCAkKCwwNDg8QERITFBUWFxgFGQQaGxwdHh8gISIjJCUmJygpKisbLC0uLzAxMjM0NTY3ODk6Ozw9Pj9AQUJDREVGR0hJSktM
AU1OT1BRUlNUVVZXWFlaW1xdXl9gYWJjZGVmZ2hpamszbG1ub3BxcnN0dXZ3eFx5ent8fX40f4CBgoOEhYaHiImKi4yNjo+QkZKT
lJWWl5iZmpucnZ6foKGiFKOkpaanqKmqq6ytrq+UsF+xsrO0tbYAAAAAAAAAFQQVkBgVohZMFYIDFQASAACIDFBA63t8oS8GAMAo
Sh+vLwYAQGGx8okBCPBD/pzfwy8GAADYgPSzLwYAQHYfcmkvBgCAt4k9Yi8GAMCORrhoLwYAQFv097AvBgCAVzpTnC8GAEC371qm
LwYAgH1NqEoBKGxqvQWbLwYAgAudT6cvBgDARhMbuC8GAADR71x6AQgsjHiZTy8GAIAiS2SHAQgsMb7LXC8GAEAaceF8AQhM7wTk
oi8GAIAn/mV4LwYAQOQ55YsBCGwgbh5lLwYAAIsRgnkvBgBAPp+jdy8GAABijwqXAQgMemFKzAEIDNWcwbkBCAypiNPsAQgQh10X
ki8BcAimVFoBCAghgBYlAAjlfTYBkAxAw2rwBXAIQQOiBVBISJuoTS8GAMDoOItsLwYAgINhUgUYCAoVaQH4kAAJqvWjLwYAABS6
DlcvBgDAAFkrey8GAIBod59hLwYAwIzx9bgBCAgsXc0hOBAAjrTt0AEQCH3jSQHwDMC/6bwBGAxAJ7fNIUAsgOVS7KwvBgBAvOMs
AWgwgAmLeXgvBgCAJn9KUAEIDBNtt7wBCKhdAsLNLwYAwGTXLLovBgCAOKnFmC8GAECBLNhtLwYAwJLUCXEvBgAADlvJISAwwFab
3VMvBgAAfx4OUQEICOhcoQHwDMB7snIhoAwAABDOBWgM828fnwEgSFT405ovBgDArCsLvS8GAICAWsJBGAzAD0GsAUgQgOXvTKoB
GAyzRzCSAQjIbUTKgi8GAECE+BiQLwYAwCsE3ZMvBgBA9cmEbi8GAMCrWaW0LwYAwNDWjK0vBgCA/5j4IcgswKpxcrIvBgAAIFbc
QVAMwOWNiyEIDAAJw34BcAyA3VgPAVgsAMUolb8vBgDAXBzyBYAIBJpsIUgsQACxyFYvBgCAIVPlAbgMgG3gnEHIEACGJu2oARgo
uEzlVS8GAEDE590BGGyA03mWZi8GAMAVJMlILwYAQD9dYWQvBgDANAlBIXAQQO0Ll1kBECzEWDvFLwYAAEGqSLQBCCxrvVW/LwYA
wMakybEBCIhwAiuFLwYAQHBGIEUvBgCAE3pHti8GAECyL8B+LwYAAPQaPgFwTED8TYyuLwYAwAvHC6QvBgAAiIWDAYAMQHUU60FA
DID7J4wBuDBAoYcuxC8GAMBr1S7HAQgMaNWkgQEICIu6uQHALIDh8+dGLwYAQPGJpUXQCE9CDQEgTEApSqRALwYAwEPU5XAvBgAA
E7i7QXgMQLJXByX4KAShuV8vBgDADkktQcAsQIT1U2cvBgAAVOzaAZgMAJTXQ0U4SFBSzmQvBgBA8E4WXS8GAACoC91hCAxA/fK3
RRgIMGk3AaAMQDu5SwGw0IDkr9yDLwYAQGnr/0kvBgAAhBDEgC8GAEAcndeeLwYAwK/FJ5ovBgCAojO5ly8GAMBko9pZAQgIpKkD
AbgQgBlcxGABEAhRHBwBQCyAr6HjdC8GAECzQ40BCDCAbma+fy8GAMDx9p1bAQgIqn4CZRgQUbE/SC9B8AQuKwFILAAUt1JrLwYA
QA/HGmV4EMFjzHYvAZAIzDe1AQgIl42uBagooXW0wy8GAEBiAj0hgAzAqXNpISAMAJMoOwEgDAAUmEhhMCxA70s+1C8GAAB0o6YB
WAyAXx3MJfgoy3HrlS8GAEDW6fYhiEzAlmjlyi8GAICeg6GpLwYAQFMV2wVgCE6x0CGgLMAtUpBeLwYAgJFcvwFgDEBHF/wF0AhX
swYhKAzALxnCIUgQAMEde4QBKAgRSV5h4CwAOFH6pS8GAMAnmFEBkAzAggcbBYAs2/KeTC8GAAD4DLRNAQgIv6ozYVgMAL9ULSWo
LPZ5G1QvBgDAl37g2wEIDJH+WawBCJiimuk9LwYAQFAu46AvBgAACyXaby8GAEC/8D15LwYAQK7JtXAvBgAVABWiAxWqAywVggMV
EBUGFQYcGAgAqYjT7C8GABgIwKKa6T0vBgAWACgIAKmI0+wvBgAYCMCimuk9LwYAEREAAADRAfDQAwAAAIIDAQgzAAECAwQFBgcI
CQoLDA0ODxAREhMUFRYXGBkaGxwdHh8gISIjJCUmJygpKissLS4vMDEyMzQ1Njc4OTo7PD0+P0BBQkNERUZHSElKS0xNTk9QUVJT
VFVWV1hZWltcXV5fYGFiY2RlZmdoaWprbG1ub3BxcnN0dXZ3eHl6e3x9fn+AgYKDhIWGh4iJiouMjY6PkJGSk5SVlpeYmZqbnJ2e
n6ChoqOkpaanqKmqq6ytrq+wsbKztLW2t7i5uru8vb6/wAAAAAAAAAAVBBlsNQAYBnNjaGVtYRUKABUEJQIYCGV2ZW50X2lkABUE
JQIYB3VzZXJfaWQAFQwlAhgKZXZlbnRfdHlwZSUATBwAAAAVBCUCGApldmVudF90aW1lJRRMjBIcLAAAAAAAFQQlAhgLaW5nZXN0
ZWRfYXQlFEyMEhwsAAAAAAAWggMZHBlcJgAcFQQZNQAGEBkYCGV2ZW50X2lkFQIWggMWuBsWqhAmhAwmCBwYCB8lAAAAAAAAGAgp
IwAAAAAAABYAKAgfJQAAAAAAABgIKSMAAAAAAAAREQAZLBUEFQAVAgAVABUQFQIAPCkGGSYAggMAAAAmABwVBBk1AAYQGRgHdXNl
cl9pZBUCFoIDFuIIFsgGJrATJrIQHBgIKAAAAAAAAAAYCAEAAAAAAAAAFgAoCCgAAAAAAAAAGAgBAAAAAAAAABERABksFQQVABUC
ABUAFRAVAgA8KQYZJgCCAwAAACYAHBUMGTUABhAZGApldmVudF90eXBlFQIWggMW0AIW1gIm/Bcm+hYcNgAoBnNlYXJjaBgLYWRk
X3RvX2NhcnQREQAZLBUEFQAVAgAVABUQFQIAPBb4GRkGGSYAggMAAAAmABwVBBk1AAYQGRgKZXZlbnRfdGltZRUCFoIDFrgbFtIa
JvQvJtAZHBgIQLUvSsQvBgAYCIDmdWU9LwYAFgAoCEC1L0rELwYAGAiA5nVlPS8GABERABksFQQVABUCABUAFRAVAgA8KQYZJgCC
AwAAACYAHBUEGTUABhAZGAtpbmdlc3RlZF9hdBUCFoIDFtgcFvIaJuZKJqI0HBgIAKmI0+wvBgAYCMCimuk9LwYAFgAoCACpiNPs
LwYAGAjAoprpPS8GABERABksFQQVABUCABUAFRAVAgA8KQYZJgCCAwAAABb6XhaCAyYIFoxPABkcGAxBUlJPVzpzY2hlbWEY2AMv
Ly8vLzFnQkFBQVFBQUFBQUFBS0FBd0FCZ0FGQUFnQUNnQUFBQUFCQkFBTUFBQUFDQUFJQUFBQUJBQUlBQUFBQkFBQUFBVUFBQURz
QUFBQXFBQUFBSFFBQUFBNEFBQUFCQUFBQURqLy8vOEFBQUVLRUFBQUFCd0FBQUFFQUFBQUFBQUFBQXNBQUFCcGJtZGxjM1JsWkY5
aGRBRE8vLy8vQUFBQ0FHai8vLzhBQUFFS0VBQUFBQ1FBQUFBRUFBQUFBQUFBQUFvQUFBQmxkbVZ1ZEY5MGFXMWxBQUFBQUFZQUNB
QUdBQVlBQUFBQUFBSUFvUC8vL3dBQUFRVVFBQUFBSUFBQUFBUUFBQUFBQUFBQUNnQUFBR1YyWlc1MFgzUjVjR1VBQUFRQUJBQUVB
QUFBMFAvLy93QUFBUUlRQUFBQUdBQUFBQVFBQUFBQUFBQUFCd0FBQUhWelpYSmZhV1FBdlAvLy93QUFBQUZBQUFBQUVBQVVBQWdB
QmdBSEFBd0FBQUFRQUJBQUFBQUFBQUVDRUFBQUFDUUFBQUFFQUFBQUFBQUFBQWdBQUFCbGRtVnVkRjlwWkFBQUFBQUlBQXdBQ0FB
SEFBZ0FBQUFBQUFBQlFBQUFBQT09ABggcGFycXVldC1jcHAtYXJyb3cgdmVyc2lvbiAyNS4wLjEZXBwAABwAABwAABwAABwAAADR
BAAAUEFSMQ==
""",
    "load_runs": """
UEFSMRUEFXAVUEwVDhUAEgAAOAQBAAkBAAIJBwQAAw0IAAQNCAAFDQg8BgAAAAAAAAAHAAAAAAAAABUAFRYVGiwVDhUQFQYVBhwY
CAcAAAAAAAAAGAgBAAAAAAAAABYAKAgHAAAAAAAAABgIAQAAAAAAAAAREQAAAAsoAgAAAA4BAwOIxhoVBBVwFWxMFQ4VABIAADgw
ACiuZVIvBgAAiIWDZgEIDOhcoXoBCAxINL+OAQgMGn4qjwEIOKgL3aIvBgAACOP6ti8GABUAFRYVGiwVDhUQFQYVBhwYCAAI4/q2
LwYAGAgAKK5lUi8GABYAKAgACOP6ti8GABgIACiuZVIvBgAREQAAAAsoAgAAAA4BAwOIxhoVBBk8NQAYBnNjaGVtYRUEABUEJQIY
BnJ1bl9pZAAVBCUCGAhydW5fdGltZSUUTIwSHCwAAAAAABYOGRwZLCYAHBUEGTUABhAZGAZydW5faWQVAhYOFqACFoQCJnQmCBwY
CAcAAAAAAAAAGAgBAAAAAAAAABYAKAgHAAAAAAAAABgIAQAAAAAAAAAREQAZLBUEFQAVAgAVABUQFQIAPCkGGSYADgAAACYAHBUE
GTUABhAZGAhydW5fdGltZRUCFg4WoAIWoAImlAMmjAIcGAgACOP6ti8GABgIACiuZVIvBgAWACgIAAjj+rYvBgAYCAAormVSLwYA
EREAGSwVBBUAFQIAFQAVEBUCADwpBhkmAA4AAAAWwAQWDiYIFqQEABkcGAxBUlJPVzpzY2hlbWEYgAIvLy8vLzdnQUFBQVFBQUFB
QUFBS0FBd0FCZ0FGQUFnQUNnQUFBQUFCQkFBTUFBQUFDQUFJQUFBQUJBQUlBQUFBQkFBQUFBSUFBQUJNQUFBQUJBQUFBTXovLy84
QUFBRUtFQUFBQUNBQUFBQUVBQUFBQUFBQUFBZ0FBQUJ5ZFc1ZmRHbHRaUUFBQmdBSUFBWUFCZ0FBQUFBQUFnQVFBQlFBQ0FBR0FB
Y0FEQUFBQUJBQUVBQUFBQUFBQVFJUUFBQUFJQUFBQUFRQUFBQUFBQUFBQmdBQUFISjFibDlwWkFBQUNBQU1BQWdBQndBSUFBQUFB
QUFBQVVBQUFBQUFBQUFBABggcGFycXVldC1jcHAtYXJyb3cgdmVyc2lvbiAyNS4wLjEZLBwAABwAAABeAgAAUEFSMQ==
""",
}

_data_dir = pathlib.Path("data") / "003-incremental-load-late-events"
_data_dir.mkdir(parents=True, exist_ok=True)
for _name, _encoded in _DATA.items():
    (_data_dir / f"{_name}.parquet").write_bytes(base64.b64decode(_encoded))

events = spark.read.parquet((_data_dir / "events.parquet").resolve().as_posix())
events.createOrReplaceTempView("events")
load_runs = spark.read.parquet((_data_dir / "load_runs.parquet").resolve().as_posix())
load_runs.createOrReplaceTempView("load_runs")

for _name in _DATA:
    print(f"{_name}: {spark.table(_name).count()} rows")

In [ ]:
#@title Grader (run this cell, no need to read it)
"""Result grading shared by the build script and every generated notebook.

build_problem.py pastes this file into the notebook verbatim, so it must stay
standalone: standard library only, no imports from the rest of the toolkit.

A result is reduced to a "summary": its columns and types, its row count and
SHA-256 fingerprints of its normalized rows. Only the summary of the correct
answer is shipped in a notebook, never the rows themselves.
"""
import datetime
import decimal
import hashlib
import json
import math

FLOAT_DECIMALS = 6


def normalize_value(value):
    """Turn one cell value into something json.dumps renders the same way everywhere."""
    if value is None or isinstance(value, (bool, int, str)):
        return value
    if isinstance(value, float):
        if math.isnan(value):
            return "NaN"
        if math.isinf(value):
            return "Infinity" if value > 0 else "-Infinity"
        return round(value, FLOAT_DECIMALS) + 0.0  # + 0.0 turns -0.0 into 0.0
    if isinstance(value, decimal.Decimal):
        return format(value, "f")
    if isinstance(value, (datetime.datetime, datetime.date)):
        return value.isoformat()
    if isinstance(value, (bytes, bytearray)):
        return bytes(value).hex()
    if hasattr(value, "asDict"):  # pyspark Row (struct column)
        return {key: normalize_value(item) for key, item in value.asDict().items()}
    if isinstance(value, dict):  # map column
        pairs = [[normalize_value(key), normalize_value(item)] for key, item in value.items()]
        return sorted(pairs, key=_encode)
    if isinstance(value, (list, tuple)):
        return [normalize_value(item) for item in value]
    return str(value)


def _encode(value):
    return json.dumps(value, sort_keys=True, separators=(",", ":"))


def _digest(lines):
    sha = hashlib.sha256()
    for line in lines:
        sha.update(line.encode("utf-8"))
        sha.update(b"\n")
    return sha.hexdigest()


def describe_schema(df):
    """Column names and Spark types, sorted by name so column order is ignored."""
    return sorted([field.name, field.dataType.simpleString()] for field in df.schema.fields)


def summarize_rows(columns, rows, order_matters=False):
    """columns: sorted [name, type] pairs. rows: normalized values in that column order."""
    encoded = [_encode(row) for row in rows]
    if not order_matters:
        encoded.sort()
    column_fingerprints = {}
    for index, (name, _) in enumerate(columns):
        column_fingerprints[name] = _digest(sorted(_encode(row[index]) for row in rows))
    return {
        "columns": [list(column) for column in columns],
        "row_count": len(rows),
        "order_matters": order_matters,
        "fingerprint": _digest([_encode(columns)] + encoded),
        "column_fingerprints": column_fingerprints,
    }


def summarize(df, order_matters=False):
    columns = describe_schema(df)
    names = [name for name, _ in columns]
    # collect() returns `timestamp` values in the machine's local time zone, which differs
    # between the build machine and Colab. Casting to string inside Spark uses the session
    # time zone instead (UTC in both places). `timestamp_ntz` columns are not affected.
    expressions = [
        "CAST(`%s` AS STRING) AS `%s`" % (name, name) if kind == "timestamp" else "`%s`" % name
        for name, kind in columns
    ]
    collected = df.selectExpr(*expressions).collect()
    rows = [[normalize_value(row[name]) for name in names] for row in collected]
    return summarize_rows(columns, rows, order_matters)


def grade(df, expected):
    """Compare a DataFrame to the expected summary. Returns (passed, hints).

    The hints say what kind of thing is wrong (columns, types, row count, which
    column) but never what the right values are.
    """
    if not hasattr(df, "schema") or not hasattr(df, "collect"):
        return False, ["check() needs a Spark DataFrame, got %s." % type(df).__name__]

    names = [field.name for field in df.schema.fields]
    duplicates = sorted({name for name in names if names.count(name) > 1})
    if duplicates:
        return False, ["Your result has duplicate column names: %s." % ", ".join(duplicates)]

    actual = dict(describe_schema(df))
    wanted = dict(expected["columns"])
    hints = []
    missing = sorted(set(wanted) - set(actual))
    extra = sorted(set(actual) - set(wanted))
    if missing:
        hints.append("Missing columns: %s." % ", ".join(missing))
    if extra:
        hints.append("Unexpected columns: %s." % ", ".join(extra))
    for name in sorted(set(wanted) & set(actual)):
        if wanted[name] != actual[name]:
            hints.append("Column %s has type %s, expected %s." % (name, actual[name], wanted[name]))
    if hints:
        return False, hints

    row_count = df.count()
    if row_count != expected["row_count"]:
        return False, [
            "Row count is wrong: your result has %d rows, expected %d."
            % (row_count, expected["row_count"])
        ]

    summary = summarize(df, expected["order_matters"])
    if summary["fingerprint"] == expected["fingerprint"]:
        return True, []

    wrong = sorted(
        name
        for name, fingerprint in summary["column_fingerprints"].items()
        if fingerprint != expected["column_fingerprints"][name]
    )
    if wrong:
        return False, ["Columns and row count are right, but values are wrong in: %s." % ", ".join(wrong)]
    if expected["order_matters"]:
        return False, ["The values are right, but the rows are in the wrong order."]
    return False, ["Each column has the right set of values, but they are combined into the wrong rows."]


def make_check(expected):
    def check(df):
        passed, hints = grade(df, expected)
        if passed:
            print("Correct. Your result matches the expected output.")
        else:
            print("Not correct yet.")
            for hint in hints:
                print("  - " + hint)
        return passed

    return check

EXPECTED = {'column_fingerprints': {'event_id': '511bdf31a2e8253b6d9639d3bd85867893493086e67014f8cefa63e340021d8a',
                         'run_id': '00d05a6894a7e463f0ec5971aef83f877956b30defc5ed9d12ecd7f9524de09b'},
 'columns': [['event_id', 'bigint'], ['run_id', 'bigint']],
 'fingerprint': '975faba51cc86f52104d5b5029c2b0eb461005991aa59b1df765e40c33950be5',
 'order_matters': False,
 'row_count': 162}

check = make_check(EXPECTED)

## The situation

A nightly job copies new rows from an event feed into the warehouse. To pick up only
new rows, it filters on `event_time > time of the previous run`.

Some events reach the feed hours or days after they happened. By then a later run has
already moved past their `event_time`, so they are never loaded. The warehouse is
missing events and nobody gets an error.

You are rebuilding the load logic. For every run in the history, work out which
events that run should have loaded.

## Input tables

### `events`

One row per delivery of an event to the feed. An event is normally delivered once,
but the feed sometimes delivers the same event again later.

| Column | Type | Description |
|---|---|---|
| `event_id` | BIGINT | Identifies the event. Repeated if the event was delivered again. |
| `user_id` | BIGINT | The user who triggered the event. |
| `event_type` | STRING | `page_view`, `search`, `add_to_cart` or `purchase`. |
| `event_time` | TIMESTAMP_NTZ | When the event happened. |
| `ingested_at` | TIMESTAMP_NTZ | When this delivery arrived in the feed. Always later than `event_time`. |

### `load_runs`

One row per run of the load job.

| Column | Type | Description |
|---|---|---|
| `run_id` | BIGINT | Unique per run. |
| `run_time` | TIMESTAMP_NTZ | The cutoff of the run. The run can see everything that had arrived by this time. |

## Requirement

- A run loads the events that arrived after the previous run's `run_time` and up to
  and including its own `run_time`. The first run loads everything that had arrived
  by its `run_time`.
- "Arrived" means `ingested_at`. When the event happened does not matter.
- Each event is loaded exactly once, by the run that covers its first delivery.
  A later delivery of the same event is ignored.
- Events that arrived after the last run are not loaded yet and are not in the result.
- A run with nothing to load has no rows in the result.

## Expected output

One row per loaded event.

| Column | Type | Description |
|---|---|---|
| `event_id` | BIGINT | |
| `run_id` | BIGINT | The run that loads the event. |

In [ ]:
events.show(5)
load_runs.show(5)

## Your answer

Use the DataFrame API. The tables are available as DataFrames: `events`, `load_runs`.

In [ ]:
result = None  # Replace None with your DataFrame.

In [ ]:
check(result)